# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nIdentifier: {getattr(metadata, 'identifier', None)}")
print(f"License: {getattr(metadata, 'license', None)}")
print(f"Number of record sets: {len(getattr(metadata, 'recordSet', [])) if hasattr(metadata, 'recordSet') else 0}")

## 2. Data Overview
Review available record sets, their fields, and `@id` references.
`mlcroissant`'s API exposes record sets and their schema for exploration.

In [ ]:
# Re-fetch metadata if needed, and find all available record sets with their @id and fields.
record_sets = getattr(metadata, 'recordSet', []) if hasattr(metadata, 'recordSet') else []

for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    fields = rs.get('field', [])
    if fields and isinstance(fields, dict):
        # Single field, wrap in list
        fields = [fields]
    print("  Fields:")
    for f in fields:
        print(f"    - @id: {f['@id']}, name: {f.get('name', 'N/A')}, dataType: {f.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from all record sets into pandas DataFrames for analysis. Reference record set and field `@id`s from above.

In [ ]:
# Extract data from each record set into a DataFrame
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display DataFrame columns for each record set
for rsid, df in dataframes.items():
    print(f"\nDataFrame for Record Set @id: {rsid}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, or grouping data by key attributes.
All columns and groupings should use column/field `@id` references.

In [ ]:
# Example: Choose the first available record set and a numeric field by its @id
from IPython.display import display
import numpy as np

if record_sets:
    main_rs = record_sets[0]
    main_rs_id = main_rs['@id']
    df = dataframes[main_rs_id]
    fields = main_rs.get('field', [])
    if fields and isinstance(fields, dict):
        fields = [fields]

    # Attempt to autodetect a numeric field (Integer/Float/Number)
    numeric_field_id = None
    numeric_types = set(["schema:Float", "schema:Integer", "schema:Number", "Float", "Integer", "Number"])
    for f in fields:
        dt = f.get('dataType')
        if dt and (dt in numeric_types or (isinstance(dt, dict) and dt.get('@id') in numeric_types)):
            numeric_field_id = f['@id']
            break

    if numeric_field_id is not None and numeric_field_id in df.columns:
        print(f"Numeric field detected: {numeric_field_id}")
        # Drop missing or non-numeric, if any
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        # Use a low threshold (mean or median?)
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 1
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())
        # Try to group by another field (categorical)
        group_field = None
        for f in fields:
            if f['@id'] != numeric_field_id:
                group_field = f['@id']
                break
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No suitable numeric field found in the first record set.")
else:
    print("No record sets found in the dataset schema.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

All axes/labels should reference `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of detected numeric field, if available
if record_sets and numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field detected, boxplot numeric across group
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, explore, and visualize data from a Croissant-defined FAIR^2 medical dataset. All schema entities—including record sets and fields—are referenced by their `@id`, improving reproducibility and transparency. 

Further steps might include deeper data analysis, hypothesis testing, or model training, all while using the rich metadata and schema provided by mlcroissant and the Croissant schema standard.